# Iterative Search for Layer Dependencies

## Overview

This notebook will iterate over all web content in the org by [items/item type](https://developers.arcgis.com/rest/users-groups-and-items/items-and-item-types.htm). Items will be stored in a list, then iteratively search for any items (web maps, apps) which are dependent upon a layer. This is a great tool to use prior to removing or replacing a layer, to ensure that you've addressed anywhere this layer shows up.

In brief, this notebook will:
1. Iterate over all web content in your org by [items/item type](https://developers.arcgis.com/rest/users-groups-and-items/items-and-item-types.htm)

    1. Iterate over all web maps in your org
    2. Iterate over all feature layers in your org
    3. Iterate over all web apps in your org
    4. Iterate over other items in org

2. Create a list of service ItemIDs

3. Create a list of service URLs

4. Iterate through list by Service ItemIDs

    1. Find the associated service URL
    2. Iterate over all maps in your org, looking for your service in:
        - Operational layers
        - Basemap layers
        - Anywhere else (search widget, etc.)
    
Matches will be printed out for user review.

## Setup

#### Run this cell to connect to your GIS and get started:

In [1]:
from arcgis.gis import GIS
import pandas as pd
import os

gis = GIS("home")

### All Org Content
First, we'll be getting a list of **all** the content in the org. using `query=''` and `max_items=-1`

#### Searching outside your org
Add the parameter `outside_org=True` to the **search** to look for maps from *any* AGOL organization that are shared publicly. Obviously, that's going to be a LOT of information to look through, so only do that if you're certain you want to, then go get some coffee or something while it executes.

In [12]:
full_org = gis.content.search(query='',max_items=-1,outside_org=False)

In [13]:
len(full_org)

1916

In [23]:
content_list = [f for f in full_org]
content_list

[<Item title:"Beta 2 - Commerce City Capital Improvement Projects" type:Web Map owner:C3_GIS_Admin>,
 <Item title:"Printable Area Map Template" type:Code Attachment owner:C3_GIS_Admin>,
 <Item title:"RestaurantsCC" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"SO_Restricted_Parcels_App" type:Code Attachment owner:C3_GIS_Admin>,
 <Item title:"PR_OpLrys_2018" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"Line" type:Shapefile owner:jallen_c3>,
 <Item title:"Liquor and Marijuana Licenses" type:Code Attachment owner:C3_GIS_Admin>,
 <Item title:"Parcel (ADCO)" type:Shapefile owner:jallen_c3>,
 <Item title:"MarijuanaEligibleParcels" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"Retail Market Potential" type:PDF owner:bphetteplace_c3>,
 <Item title:"Northern Range Trade Area" type:PDF owner:bphetteplace_c3>,
 <Item title:"Commerce City, Colorado" type:PDF owner:bphetteplace_c3>,
 <Item title:"Community Profile 37" type:PDF owner:bphetteplace_c3>,
 <Item title:"Com

In [24]:
len(content_list)

1916

#### Reviewing Content Types
Using `.type` and the `set()` function to generate a list of all possible content types found in the Organization.

In [25]:
tLst = [t.type for t in content_list]
list(set(tLst))

['File Geodatabase',
 'Tile Package',
 'Solution',
 'Shapefile',
 'Workforce Project',
 'Geoprocessing Service',
 'Hub Page',
 'Mobile Map Package',
 'Desktop Application Template',
 'Hub Initiative',
 'Image',
 'QuickCapture Project',
 'Web Experience',
 'Network Analysis Service',
 'StoryMap',
 'StoryMap Theme',
 'PDF',
 'Feature Service',
 'Administrative Report',
 'Service Definition',
 'Hub Site Application',
 'Map Service',
 'CAD Drawing',
 'Geocoding Service',
 'Web Scene',
 'Web Map',
 'Web Experience Template',
 'Form',
 'GeoJson',
 'Layer Package',
 'Notebook',
 'Document Link',
 'Native Application',
 'CSV',
 'Operation View',
 'Feature Collection',
 'Application',
 'Map Package',
 'CSV Collection',
 'Code Attachment',
 'Microsoft Excel',
 'KML',
 'Report Template',
 'Dashboard',
 'Image Service',
 'Project Package',
 'WMS',
 'Web Mapping Application']

### Other Content Types

Changing the parameter `item_type=` to the various types of content found in AGOL organization.

In [28]:
solution = gis.content.search(query='', item_type='Solution', max_items=-1, outside_org=False)
solution

[<Item title:"Traffic Crash Analysis" type:Solution owner:rwoolf_c3>,
 <Item title:"Share Open dummy open data junk" type:Solution owner:C3_GIS_Admin>,
 <Item title:"My Trash Services" type:Solution owner:jmoening_c3>,
 <Item title:"Special Event Operations" type:Solution owner:jmoening_c3>,
 <Item title:"Traffic Crash Analysis" type:Solution owner:jmoening_c3>,
 <Item title:"Commerce City GIS" type:Solution owner:mcampe_c3>,
 <Item title:"Traffic Crash Analysis" type:Solution owner:jmoening_c3>]

In [29]:
fGDB = gis.content.search(query='', item_type='File Geodatabase', max_items=-1, outside_org=False)
fGDB

[<Item title:"OGP_MAP_gdb" type:File Geodatabase owner:dmartinelli_c3>,
 <Item title:"test_AOIs" type:File Geodatabase owner:aolson_c3>,
 <Item title:"DOLA_Districts_gdb" type:File Geodatabase owner:jmoening_c3>,
 <Item title:"BMP_Inspection_2021_Rev2_download_102921" type:File Geodatabase owner:C3_GIS_Admin>,
 <Item title:"S123_d430e76892404dc6af2be49e1832ae45_FGDB" type:File Geodatabase owner:mcampe_c3>,
 <Item title:"Parks_Data" type:File Geodatabase owner:jallen_c3>,
 <Item title:"Parcel_Data_April_2024" type:File Geodatabase owner:jmoening_c3>,
 <Item title:"GDB for CIPP Tracking Report - v2" type:File Geodatabase owner:jmoening_c3>,
 <Item title:"CC_TaxEntities_v2_gdb" type:File Geodatabase owner:jmoening_c3>,
 <Item title:"UtiliSyncDemo_gdb" type:File Geodatabase owner:jallen_c3>,
 <Item title:"GDB for CIPP Tracking Report" type:File Geodatabase owner:aolson_c3>,
 <Item title:"PriorityCorridors.gdb" type:File Geodatabase owner:C3_GIS_Admin>,
 <Item title:"CIP Web Map" type:File 

In [30]:
featService = gis.content.search(query='', item_type='Feature Service', max_items=-1, outside_org=False)
featService

[<Item title:"SpecialEventAssets" type:Feature Layer Collection owner:jmoening_c3>,
 <Item title:"Wards_buffer" type:Feature Layer Collection owner:rwoolf_c3>,
 <Item title:"CIP_02_Roads_pub" type:Feature Layer Collection owner:C3_GIS_Admin>,
 <Item title:"test_AOIs" type:Feature Layer Collection owner:aolson_c3>,
 <Item title:"Commerce City Parcels" type:Feature Layer Collection owner:mcampe_c3>,
 <Item title:"Duty Maintain Web Map_WFL1" type:Feature Layer Collection owner:rwoolf_c3>,
 <Item title:"CIP_06_BasemapLines_pub" type:Feature Layer Collection owner:C3_GIS_Admin>,
 <Item title:"c3_History" type:Feature Layer Collection owner:jallen_c3>,
 <Item title:"CIP_04_Rec_pub" type:Feature Layer Collection owner:C3_GIS_Admin>,
 <Item title:"CC Encampment Location" type:Feature Layer Collection owner:rwoolf_c3>,
 <Item title:"Police Patrol Beat" type:Feature Layer Collection owner:mcampe_c3>,
 <Item title:"CommunityAddresses" type:Feature Layer Collection owner:C3_GIS_Admin>,
 <Item titl

In [31]:
webMapServ = gis.content.search(query='', item_type='WMS', max_items=-1, outside_org=False)
webMapServ

[<Item title:"Assisted Living (DRCOG)" type:WMS owner:C3_GIS_Admin>]

In [32]:
servMap = gis.content.search(query='', item_type='Map Service', max_items=-1, outside_org=False)
servMap

[<Item title:"RestaurantsCC" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"PR_OpLrys_2018" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"MarijuanaEligibleParcels" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"CC_Cartegraph_BasemapTEST" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"Current Land Use" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"Active Development Under Review Map Service" type:Map Image Layer owner:aolson_c3>,
 <Item title:"IGA Growth Boundary" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"StormWaterIntets_Only3" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"SNOW PLOW ROUTES" type:Map Image Layer owner:mcampe_c3>,
 <Item title:"Commerce City Historic Map" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"pub_OP_CIP_non2K_Points" type:Map Image Layer owner:C3_GIS_Admin>,
 <Item title:"Mowing Schedule Map" type:Map Image Layer owner:aolson_c3>,
 <Item title:"Buffalo Run Golf Course Basemap Feature

In [33]:
imgServ = gis.content.search(query='', item_type='Image Service', max_items=-1, outside_org=False)
imgServ

[<Item title:"Aerial 2014 DRAPP" type:Imagery Layer owner:C3_GIS_Admin>,
 <Item title:"Denver Regional Aerial Photography - 2012, Commerce City Area" type:Imagery Layer owner:C3_GIS_Admin>,
 <Item title:"Commerce_City_DRAPP_2018" type:Imagery Layer owner:C3_GIS_Admin>,
 <Item title:"DRAPP2018CommerceCity" type:Imagery Layer owner:C3_GIS_Admin>]